In [ ]:
import os
import requests
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from dotenv import load_dotenv

# ==========================================
# 0. 환경 변수(.env) 로드 및 API 인증키 설정
# ==========================================
load_dotenv()
SERVICE_KEY = os.getenv("DATA_GO_KR_KEY")

# ==========================================
# 1. 공휴일 API 연동 (2023년 ~ 2026년)
# ==========================================
print("공공데이터포털에서 공휴일 데이터를 수집 중입니다...")
holiday_set = set()
url = "http://apis.data.go.kr/B090041/openapi/service/SpcdeInfoService/getRestDeInfo"

target_periods = []
for year in range(2023, 2027):
    for month in range(1, 13):
        if year == 2026 and month > 3:
            break
        target_periods.append((year, month))

if SERVICE_KEY:
    for year, month in target_periods:
        params = {
            'serviceKey': SERVICE_KEY,
            'solYear': str(year),
            'solMonth': f"{month:02d}",
            'numOfRows': '100'
        }
        try:
            response = requests.get(url, params=params)
            if response.status_code == 200:
                root = ET.fromstring(response.content)
                items = root.findall('.//item')
                for item in items:
                    is_holiday = item.find('isHoliday')
                    locdate = item.find('locdate')
                    if is_holiday is not None and is_holiday.text == 'Y' and locdate is not None:
                        holiday_set.add(locdate.text)
        except Exception as e:
            print(f"경고: {year}년 {month}월 공휴일 데이터를 가져오는 중 오류 발생 - {e}")

# ==========================================
# 2. 메인 데이터(Oworld.csv) 파일 로드 및 날짜 복원
# ==========================================
if not os.path.exists('Oworld.csv'):
    raise FileNotFoundError("Oworld.csv 파일이 존재하지 않습니다.")

df = pd.read_csv('Oworld.csv', encoding='utf-8-sig')
total_len = len(df)

day_cols = ['mon', 'tue', 'wed', 'thur', 'fri', 'sat', 'sun']
if all(col in df.columns for col in day_cols):
    day_matrix = df[day_cols].values
    weekday_indices = np.argmax(day_matrix, axis=1)
    
    base_dates = pd.date_range(start='2023-01-01', periods=len(df), freq='D')
    first_row_weekday = weekday_indices[0]
    base_start_weekday = base_dates[0].weekday()
    offset = (first_row_weekday - base_start_weekday) % 7
    
    df['parsed_date'] = pd.date_range(
        start=pd.to_datetime('2023-01-01') + pd.Timedelta(days=int(offset)), 
        periods=len(df), freq='D'
    )
    print("메인 데이터 요일 컬럼 패턴 분석 및 날짜 복원 완료.")
else:
    raise ValueError("메인 데이터에 필수 요일 컬럼(mon~sun)이 누락되어 있습니다.")

df['holiday'] = df['parsed_date'].dt.strftime('%Y%m%d').isin(holiday_set).astype(int)

# ==========================================
# 3. 샌드위치 데이(Sandwich Day) 탐지 로직 추가
# ==========================================
# 휴일 또는 주말인 날을 쉬는 날(non_working)로 정의
df['non_working'] = ((df['weekend'] == 1) | (df['holiday'] == 1)).astype(int)

# 전날과 다음날이 쉬는 날인지 확인
df['prev_non_working'] = df['non_working'].shift(1).fillna(0)
df['next_non_working'] = df['non_working'].shift(-1).fillna(0)

# 샌드위치 데이 조건: 평일(weekend==0)이고, 본인은 공휴일이 아니며(holiday==0), 전날과 다음날 중 적어도 하나 이상이 쉬는 날인 경우
df['sandwich'] = (
    (df['weekend'] == 0) & 
    (df['holiday'] == 0) & 
    ((df['prev_non_working'] == 1) & (df['next_non_working'] == 1))
).astype(int)

sandwich_df = df[df['sandwich'] == 1]
print(f"\n[샌드위치 데이 분석 결과]")
print(f"- 탐지된 샌드위치 데이 일수: {len(sandwich_df)}일")
if len(sandwich_df) > 0:
    print(sandwich_df[['parsed_date', 'visitors', 'temperature', 'humidity']].head(10))

# ==========================================
# 4. 피처 전처리 (강수여부 변환 및 정규화)
# ==========================================
if 'rain' in df.columns:
    df['rain'] = (df['rain'] > 0).astype(int)

numeric_features = ['temperature', 'humidity']
valid_numeric_features = [col for col in numeric_features if col in df.columns and df[col].nunique() > 1]

if valid_numeric_features:
    scaler = MinMaxScaler()
    df[valid_numeric_features] = scaler.fit_transform(df[valid_numeric_features])

# 임시 날짜 컬럼 정리
df = df.drop(columns=['parsed_date', 'non_working', 'prev_non_working', 'next_non_working'], errors='ignore')
if 'date' in df.columns:
    df = df.drop(columns=['date'])

output_filename = 'Oworld_final_processed.csv'
df.to_csv(output_filename, index=False, encoding='utf-8-sig')
print(f"\n최종 전처리 파일 저장 완료: {output_filename}")

# ==========================================
# 5. Oworld_hold.csv 비교 분석
# ==========================================
if os.path.exists('Oworld_hold.csv'):
    print("\n[Oworld_hold.csv 보류 데이터 비교 분석]")
    df_hold = pd.read_csv('Oworld_hold.csv', encoding='utf-8-sig')
    common_cols = [col for col in ['visitors', 'temperature', 'humidity'] if col in df.columns and col in df_hold.columns]
    if common_cols:
        comparison_summary = pd.DataFrame({
            'Main_Mean': df[common_cols].mean(),
            'Hold_Mean': df_hold[common_cols].mean(),
            'Main_Std': df[common_cols].std(),
            'Hold_Std': df_hold[common_cols].std()
        })
        print(comparison_summary)

# ==========================================
# 5. 카테고리 분류 컬럼 생성 (공휴일, 주말, 샌드위치 데이, 일반 평일)
# ==========================================
# 1) 쉬는 날(주말 또는 공휴일) 정의
df['non_working'] = ((df['weekend'] == 1) | (df['holiday'] == 1)).astype(int)

# 2) 전날/다음날 쉬는 날 여부 확인
df['prev_non_working'] = df['non_working'].shift(1).fillna(0)
df['next_non_working'] = df['non_working'].shift(-1).fillna(0)

# 3) 샌드위치 데이 판정 (평일 이면서, 공휴일이 아니며, 전후가 모두 쉬는 날인 경우)
df['_day'] = (
    (df['weekend'] == 0) & 
    (df['holiday'] == 0) & 
    (df['prev_non_working'] == 1) & 
    (df['next_non_working'] == 1)
).astype(int)

sandwich_count = df['sandwich'].sum()
print(f"✨ 탐지된 샌드위치 데이 개수: {sandwich_count}개")

# 4) 통합 카테고리 컬럼 생성 (우선순위: 공휴일 > 샌드위치 데이 > 주말 > 일반 평일)
def categorize_day(row):
    if row['holiday'] == 1:
        return '공휴일'
    elif row['sandwich'] == 1:
        return '샌드위치 데이'
    elif row['weekend'] == 1:
        return '주말'
    else:
        return '일반 평일'

df['day_category'] = df.apply(categorize_day, axis=1)

# 카테고리별 평균 방문자 수 출력 확인
category_means = df.groupby('day_category')['visitors'].mean()
print("\n--- [카테고리별 평균 방문자 수 요약] ---")
print(category_means)

# ==========================================
# 6. 데이터 시각화 생성 및 저장 
# ==========================================
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1) 요일별 평균 방문자 수 비교
day_names = ['월', '화', '수', '목', '금', '토', '일']
day_means = [df[df[col] == 1]['visitors'].mean() for col in day_cols]
sns.barplot(x=day_names, y=day_means, ax=axes[0], palette='Blues_d')
axes[0].set_title('요일별 평균 방문자 수')
axes[0].set_ylabel('평균 방문자 수')
axes[0].grid(True, linestyle='--', alpha=0.5)

# 2) 주말 vs 평일 평균 방문자 수 비교
weekend_means = df.groupby('weekend')['visitors'].mean()
sns.barplot(x=['평일', '주말'], y=weekend_means.values, ax=axes[1], palette=['lightcoral', 'lightgreen'])
axes[1].set_title('평일 vs 주말 평균 방문자 수')
axes[1].set_ylabel('평균 방문자 수')
axes[1].grid(True, linestyle='--', alpha=0.5)

# 3) 공휴일 / 주말 / 샌드위치 데이 / 일반 평일 비교
category_order = ['일반 평일', '샌드위치 데이', '주말', '공휴일']
sns.barplot(x=category_order, y=[category_means.get(cat, 0) for cat in category_order], 
            ax=axes[2], palette='Set2')
axes[2].set_title('휴무 형태별 평균 방문자 수 분석')
axes[2].set_ylabel('평균 방문자 수')
axes[2].tick_params(axis='x', rotation=15)
axes[2].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

공공데이터포털에서 공휴일 데이터를 수집 중입니다...
메인 데이터 요일 컬럼 패턴 분석 및 날짜 복원 완료.

[샌드위치 데이 분석 결과]
- 탐지된 샌드위치 데이 일수: 11일
     parsed_date  visitors  temperature  humidity
147   2023-05-28      1005         20.4      89.3
273   2023-10-01     17317         16.7      65.5
280   2023-10-08      6819         17.3      70.0
357   2023-12-24      1605         -0.4      79.6
364   2023-12-31      2418          3.8      75.3
466   2024-04-11      1799         17.5      55.6
489   2024-05-04      1941         14.4      86.5
640   2024-10-02      7517         16.2      80.5
791   2025-03-02        24          3.4      70.4
1007  2025-10-04      2720         19.9      95.8

최종 전처리 파일 저장 완료: Oworld_final_processed.csv

[Oworld_hold.csv 보류 데이터 비교 분석]


KeyError: 'sandwich'